## Feature Engineering Pipeline

In [69]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as pyplot

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, PowerTransformer, FunctionTransformer

from sklearn.pipeline import Pipeline,make_pipeline

import pickle

import warnings
warnings.filterwarnings('ignore')

In [70]:
df = pd.read_csv('Procurement KPI Data.csv')

In [71]:
df.head(2)

,PO_ID,Supplier,Item_Category,Quantity,Defective_Units,Compliance,Lead_Time,Negotiated_vs_Unit_Price,Price_Ratio
0,PO-00001,Alpha_Inc,Office Supplies,1176,NaN,Yes,8.0,2.32,0.884749
1,PO-00002,Delta_Logistics,Office Supplies,1509,235.0,Yes,10.0,1.98,0.949644


## Feature Engineering & Transformation

In [72]:
df.isnull().sum()

PO_ID                         0
Supplier                      0
Item_Category                 0
Quantity                      0
Defective_Units             136
Compliance                    0
Lead_Time                    87
Negotiated_vs_Unit_Price      0
Price_Ratio                   0
dtype: int64

### Numerical Features

In [82]:
def lead_time_fill(df):

    df['Lead_Time'] = df['Lead_Time'].fillna(df['Lead_Time'].mean())

    return df

In [83]:
lead_time_fill_ft = FunctionTransformer(lead_time_fill)

In [84]:
def outliers_capping(df):

   for col in ['Quantity','Defective_Units']:
      
      q3 = df[col].quantile(0.75)
      q1 = df[col].quantile(0.25)
      iqr = q3 - q1
      lb = q1 - 1.5*iqr
      ub = q3 + 1.5*iqr

      df[col] = df[col].clip(lower=lb, upper=ub)

      return df

In [85]:
outliers_capping_ft = FunctionTransformer(outliers_capping)

In [86]:
numerical_trf = Pipeline(steps=[('transforming', PowerTransformer(method='yeo-johnson')),
                                ('scaling', StandardScaler())])                                                                                                                                             

### Categorical Features

In [87]:
encoding_cat_trf = Pipeline(steps=[('ohe',OneHotEncoder(sparse=False, handle_unknown='ignore'))])

## Pipeline Developement

In [89]:
df.head(1)

,PO_ID,Supplier,Item_Category,Quantity,Defective_Units,Compliance,Lead_Time,Negotiated_vs_Unit_Price,Price_Ratio
0,PO-00001,Alpha_Inc,Office Supplies,1176,NaN,Yes,8.0,2.32,0.884749


In [90]:
trf = ColumnTransformer([
    ('numerical_trf', numerical_trf, ['Quantity','Negotiated_vs_Unit_Price','Lead_Time']),
    ('encoding_cat_trf', encoding_cat_trf, ['Supplier', 'Item_Category', 'Compliance'])
])

In [91]:
feature_engineering_pipeline = make_pipeline(lead_time_fill_ft, outliers_capping_ft, trf)

In [92]:
feature_engineering_pipeline

Pipeline(steps=[('functiontransformer-1',
                 FunctionTransformer(func=<function lead_time_fill at 0x000001615211A2A0>)),
                ('functiontransformer-2',
                 FunctionTransformer(func=<function outliers_capping at 0x000001615211B100>)),
                ('columntransformer',
                 ColumnTransformer(transformers=[('numerical_trf',
                                                  Pipeline(steps=[('transforming',
                                                                   PowerTransformer()),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Quantity',
                                                   'Negotiated_vs_Unit_Price',
                                                   'Lead_Time']),
                                                 ('encoding_cat_trf',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse=False))]),
                                                  ['Supplier', 'Item_Category',
                                                   'Compliance'])]))])

## Save Pipeline

In [93]:
with open('feature_engineering_pipeline.pkl','wb') as f:
    pickle.dump(feature_engineering_pipeline, f)